# DeepFashion-MultiModal: captions and tags

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/deepfashion_captions.ipynb)

Every described photo of DeepFashion-MultiModal (42,544 of its 44,096) gets a caption from the stock Qwen3.5-4B, asked with `captions.FASHION_ASK` and the photo's own textual description as the helper line, and booru tags from the animetimm EVA-giant tagger (`captions.run`). The rows ship to `extractions/<stamp>/` of the dataset `AbstractPhil/qwen-thinking-images` while the run goes; after a lost runtime, set `RESUME` in C3 to the shipped folder and run C1-C3 again.

The photos and their descriptions come from [tilak1114/deepfashion](https://huggingface.co/datasets/tilak1114/deepfashion), a full-size copy on the Hub (750 x 1101, the photos' own file names) of the described photos of DeepFashion-MultiModal ([github.com/yumingj/DeepFashion-MultiModal](https://github.com/yumingj/DeepFashion-MultiModal)); the dataset's official Google Drive links refuse Colab's machines. For non-commercial research use under the dataset's agreement. A row holds the file name, our caption, our tags and how much of the caption repeats the description word for word; not the dataset's images or descriptions.

C4 fits a 3D body to every photo with SAM 3D Body (`pose`): the hips, the chest and the head as angles against the camera, the twist of the shoulders against the hips and the head against the chest, in each row (a smoke of 100 photos first). C5 is a prompt test: captions whose ask carries the fit's pose line, to compare with C3's captions of the same photos. The SAM 3D Body weights are gated: request access to [facebook/sam-3d-body-vith](https://huggingface.co/facebook/sam-3d-body-vith) with the account of the `HF_TOKEN` secret first (the SAM License).

Runtime: a GPU (the G4, RTX PRO 6000) and the `HF_TOKEN` secret with write access to the dataset repo and access to the gated animetimm tagger. Run C1, restart the session (Runtime > Restart session), then run C2 and the cell you want: C3 (captions and tags; C3b redoes the photos a finished run missed), C4 (body angles: the smoke; C4b every photo), C5 (the pose-line prompt test), C6 (the lighting trial: ShotVL-7B and CinemaCLIP on 100 photos, shipped to `raw/` of AbstractPhil/deepfashion-multimodal-recaptioned).

In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.48", "transformers>=5.0", "accelerate>=1.0", "timm>=1.0")
pip("roma", "pytorch-lightning", "yacs", "omegaconf", "einops", "braceexpand")   # SAM 3D Body's inference path (pose.NEEDS), for C4-C5

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (the run still works, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

print("installed; now Runtime > Restart session, then run C2 and C3")

In [ ]:
# C2: the photos and their descriptions, from the full-size copy on the Hub, into images.zip + captions.json (non-commercial research use)
import json, os, shutil, time, zipfile
import pyarrow.parquet as pq
from huggingface_hub import snapshot_download
from PIL import Image

DATA = "/content/deepfashion"
MIRROR = "tilak1114/deepfashion"   # 750 x 1101 photos under their own file names (img_name), their descriptions in caption
IMAGES, DESCRIPTIONS = os.path.join(DATA, "images.zip"), os.path.join(DATA, "captions.json")
os.makedirs(DATA, exist_ok=True)

if zipfile.is_zipfile(IMAGES) and os.path.exists(DESCRIPTIONS):
    print("images.zip and captions.json already here")
else:
    t0 = time.time()
    local = snapshot_download(MIRROR, repo_type="dataset", allow_patterns=["data/*.parquet"], local_dir=os.path.join(DATA, "mirror"))
    shards = sorted(os.path.join(local, "data", f) for f in os.listdir(os.path.join(local, "data")) if f.endswith(".parquet"))
    print(f"{MIRROR}: {len(shards)} shards, {sum(os.path.getsize(s) for s in shards) / 1e9:.2f} GB in {time.time() - t0:.0f} s", flush=True)
    texts, repeated, empty = {}, 0, 0
    with zipfile.ZipFile(IMAGES + ".part", "w", zipfile.ZIP_STORED) as zf:
        for s in shards:
            for b in pq.ParquetFile(s).iter_batches(batch_size=256, columns=["img_name", "caption", "image"]):
                for name, text, image in zip(*(b.column(c).to_pylist() for c in ("img_name", "caption", "image"))):
                    if name in texts:
                        repeated += 1
                    elif not (image or {}).get("bytes"):
                        empty += 1
                    else:
                        zf.writestr(name, image["bytes"])
                        texts[name] = text or ""
    os.replace(IMAGES + ".part", IMAGES)
    with open(DESCRIPTIONS, "w", encoding="utf-8") as fh:
        json.dump({k: v for k, v in texts.items() if v}, fh)
    shutil.rmtree(os.path.join(DATA, "mirror"), ignore_errors=True)
    print(f"{len(texts)} photos written ({repeated} repeated names, {empty} without image bytes skipped) in {time.time() - t0:.0f} s")

with zipfile.ZipFile(IMAGES) as zf:
    names = sorted(m for m in zf.namelist() if m.lower().endswith(".jpg"))
    size = Image.open(zf.open(names[0])).size
with open(DESCRIPTIONS, encoding="utf-8") as fh:
    print(f"{len(names)} photos, {len(json.load(fh))} descriptions; {names[0]}: {size[0]} x {size[1]}")

In [ ]:
# C3: caption and tag every photo; the rows ship to the dataset repo while the run goes
from qwen_symbolic_thinking.captions import FASHION_ASK, run

RESUME = None   # after a lost runtime: the shipped folder, e.g. "extractions/20261104_230000_deepfashion_mm_captions"
TAG_OMIT = ("short hair", "see-through", "see-through sleeves", "see-through shirt", "see-through clothes", "hair bun",
            "single hair bun", "double bun", "parted lips", "lips", "realistic", "photorealistic", "asian")   # the reviewer's leave-out list

summary = run(IMAGES, DESCRIPTIONS, model_id="Qwen/Qwen3.5-4B", ask=FASHION_ASK, batch=64, chunk=512, max_new_tokens=160,
              tagger="animetimm/eva_giant_patch14_560.dbv4-full", tag_omit=TAG_OMIT, tag_min_precision=0.4,
              tag="deepfashion_mm_captions", resume=RESUME)
print(summary["path_in_repo"])

In [ ]:
# C3b: redo the photos a finished caption run missed (its missing.jsonl, e.g. out of memory), into the same folder
from qwen_symbolic_thinking.captions import FASHION_ASK, run

RESUME = "extractions/20261105_001419_deepfashion_mm_captions"   # the finished run's folder
TAG_OMIT = ("short hair", "see-through", "see-through sleeves", "see-through shirt", "see-through clothes", "hair bun",
            "single hair bun", "double bun", "parted lips", "lips", "realistic", "photorealistic", "asian")   # as in C3

summary = run(IMAGES, DESCRIPTIONS, model_id="Qwen/Qwen3.5-4B", ask=FASHION_ASK, batch=16, chunk=512, max_new_tokens=160,
              tagger="animetimm/eva_giant_patch14_560.dbv4-full", tag_omit=TAG_OMIT, tag_min_precision=0.4,
              tag="deepfashion_mm_captions", resume=RESUME)
print(summary["path_in_repo"], summary["done_before"], summary["done_now"], summary["missing"])

In [ ]:
# C4: a 3D body fit for every photo (SAM 3D Body): the hips, chest and head angles in each row; a smoke of SMOKE photos first
# The weights are gated: request access to facebook/sam-3d-body-vith on the Hub with the HF_TOKEN account first.
import json, os, re, statistics
from qwen_symbolic_thinking import pose
from qwen_symbolic_thinking.captions import run

SMOKE = 100            # the first SMOKE photos: the time a photo and the angles by view; None: every photo
RESUME = None          # after a lost runtime: the shipped folder of the full pass
FLAT = lambda f: "_6_flat" in f      # a garment laid flat: no body to fit

summary = run(IMAGES, DESCRIPTIONS, model_id=None, tagger=None, pose_fit=pose.WEIGHTS, pose_skip=FLAT, n=SMOKE,
              tag="deepfashion_mm_pose" + (f"_smoke{SMOKE}" if SMOKE else ""), resume=RESUME)
with open(os.path.join("/content/captions", summary["path_in_repo"].split("/")[-1], "rows.jsonl"), encoding="utf-8") as fh:
    rows = [json.loads(line) for line in fh]
views = {}
for r in rows:
    view = re.search(r"_\d+_([a-z]+)\.jpg$", r["file"])
    if r.get("pose"):
        views.setdefault(view.group(1) if view else "?", []).append(abs(r["pose"]["hips"]["turn"]))
for view, turns in sorted(views.items()):   # a fit to trust: front and full near 0, side near 90, back near 180
    print(f"{view}: {len(turns)} photos, the hips' turn from the camera (median) {statistics.median(turns):.0f} degrees")
print(summary["path_in_repo"], summary["s_per_photo_by_op"], summary["pose"]["counts"])

In [ ]:
# C4b: the 3D body fit for every photo, after C4's smoke reads right (front near 0, side near 90, back near 180 degrees)
from qwen_symbolic_thinking import pose
from qwen_symbolic_thinking.captions import run

RESUME = None   # after a lost runtime: the shipped folder, e.g. "extractions/20261105_040000_deepfashion_mm_pose"
summary = run(IMAGES, DESCRIPTIONS, model_id=None, tagger=None, pose_fit=pose.WEIGHTS, pose_skip=lambda f: "_6_flat" in f,
              tag="deepfashion_mm_pose", resume=RESUME)
print(summary["path_in_repo"], summary["s_per_photo_by_op"], summary["pose"]["counts"])

In [ ]:
# C5: the prompt test: captions whose ask carries the pose line, on the first N photos (compare with C3's rows of the same files)
from qwen_symbolic_thinking import pose
from qwen_symbolic_thinking.captions import FASHION_ASK, run

N = 256
summary = run(IMAGES, DESCRIPTIONS, model_id="Qwen/Qwen3.5-4B", ask=FASHION_ASK, batch=64, chunk=512, max_new_tokens=160,
              tagger=None, pose_fit=pose.WEIGHTS, pose_ask=True, pose_skip=lambda f: "_6_flat" in f, n=N,
              tag=f"deepfashion_mm_captions_pose_ask{N}")
print(summary["path_in_repo"], summary["s_per_photo_by_op"])

In [ ]:
# C6: the lighting trial: ShotVL-7B (its benchmark's two lighting questions, open and as a choice) and CinemaCLIP (its lighting heads)
# on 100 photos: 90 drawn by view, 10 the booru tagger marked "shadow"; the rows ship to the recaptioned dataset's raw/ folder
import json, random, subprocess, sys, zipfile
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime==1.23.2"])   # CinemaCLIP's released ONNX encoder
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from qwen_symbolic_thinking import lighting

REPO = "AbstractPhil/deepfashion-multimodal-recaptioned"
QUOTA = {"front": 22, "side": 18, "back": 18, "full": 18, "additional": 14}
t = pq.read_table(hf_hub_download(REPO, "data/recaptioned.parquet", repo_type="dataset"), columns=["file", "view", "tags"]).to_pydict()
with zipfile.ZipFile(IMAGES) as zf:
    present = {m.rsplit("/", 1)[-1] for m in zf.namelist()}
rng, by_view, shadow = random.Random(1105), {}, []
for f, v, tags in zip(t["file"], t["view"], t["tags"]):
    if f in present:
        by_view.setdefault(v, []).append(f)
        if any((tg[0] if isinstance(tg, list) else tg) == "shadow" for tg in json.loads(tags or "[]")):
            shadow.append(f)
files = [f for v, n in QUOTA.items() for f in rng.sample(sorted(by_view[v]), n)]
files += rng.sample(sorted(set(shadow) - set(files)), 10)
summary = lighting.trial(IMAGES, files, out_repo=REPO, path_in_repo="raw", run_name="deepfashion_mm_lighting_trial100",
                         views=dict(zip(t["file"], t["view"])))
print(summary["stamp"], summary["s_per_photo"], "processor:", summary["processor_from"])